# pandas 00. 基本はこの10個

pandas には数百のメソッドがあるが、**普段書くのはこの10個**でほぼ足りる。

| # | やること | SQL でいうと | pandas |
| --- | --- | --- | --- |
| 1 | 読む | — | `pd.read_csv` |
| 2 | 見る | — | `head` / `shape` / `dtypes` / `value_counts` |
| 3 | 列を選ぶ | `SELECT` | `df[["a", "b"]]` |
| 4 | 行を絞る | `WHERE` | `df[条件]` |
| 5 | 列を作る | `SELECT ... AS` | `df["new"] = ...` |
| 6 | 並べる | `ORDER BY` | `sort_values` |
| 7 | まとめる | `GROUP BY` | `groupby().agg()` |
| 8 | 横にくっつける | `JOIN` | `merge` |
| 9 | 縦にくっつける | `UNION ALL` | `concat` |
| 10 | 重複を消す | `DISTINCT` | `drop_duplicates` |

最後に「書く」(`to_parquet`)が付いて終わり。

**この10個の組み合わせで、実務のコードの9割が書ける。**
残りの1割は、この10個では書けないと分かってから調べればよい。

このノートブックは説明が中心で、引っかけは無い。
細かい引数の違いは `pandas-01` 以降で扱う。

---
## 0. データ構造は2つだけ

pandas に出てくる型は、実質この2つ。

In [ ]:
import pandas as pd

# DataFrame = 表。Excel のシート1枚だと思ってよい
df = pd.read_csv("/data/sales.csv")
df

In [ ]:
# Series = 1列。DataFrame から1列取り出すとこれになる
s = df["qty"]
print(type(s))
s

| | 何か | 取り出し方 |
| --- | --- | --- |
| **DataFrame** | 表(2次元) | `pd.read_csv(...)` |
| **Series** | 1列(1次元) | `df["列名"]` |

**角括弧を2つにすると DataFrame のまま**取り出せる。

```python
df["qty"]      # Series
df[["qty"]]    # DataFrame (列が1つの表)
```

どちらも「列を取り出す」だが、返る型が違う。
計算に使うなら Series、表として扱うなら DataFrame。

左端の `0 1 2 3 ...` は **index**(行の名前)。いまは連番だが、
行を絞ったり並べ替えたりすると**飛び番になる**。気になったら
`.reset_index(drop=True)` で振り直す。

---
## 1. 読む

In [ ]:
df = pd.read_csv("/data/sales.csv")
shops = pd.read_csv("/data/shops.csv")

display(df)
shops

```python
pd.read_csv(path)        # CSV
pd.read_parquet(path)    # Parquet
pd.read_excel(path)      # Excel
pd.read_json(path)       # JSON
```

だいたい `pd.read_なんとか(パス)` で読める。

引数はたくさんあるが、**最初は何も付けなくてよい**。
付けるべき引数の話は `pandas-01` でやる。

---
## 2. 見る

**何をするにも、まずこれ。** データを見ずに書き始めると必ず外す。

In [ ]:
print("行数と列数:", df.shape)      # (行, 列)
print("列名:", df.columns.tolist())
print()
print("型:")
print(df.dtypes)

In [ ]:
display(df.head(3))      # 最初の3行
display(df.tail(3))      # 最後の3行
df.sample(3)             # ランダムに3行

In [ ]:
# この列にどんな値が入っているか
print("種類:", df["shop"].unique())
print("種類数:", df["shop"].nunique())
print()
print("値ごとの件数:")
print(df["shop"].value_counts())

In [ ]:
# 数値列の要約
df.describe()

| やりたいこと | 書き方 |
| --- | --- |
| 大きさ | `df.shape` |
| 列名 | `df.columns.tolist()` |
| 型 | `df.dtypes` |
| 最初/最後を見る | `df.head()` / `df.tail()` |
| **どんな値があるか** | **`df["col"].unique()`** |
| 値ごとの件数 | `df["col"].value_counts()` |
| 欠損の数 | `df.isna().sum()` |
| 数値の要約 | `df.describe()` |

**`unique()` と `value_counts()` を最初に叩く癖をつける。**
仕様書を読んで想像するより、データに聞いたほうが速くて確実。

---
## 3. 列を選ぶ (SELECT)

In [ ]:
df[["shop", "item", "qty"]]

```sql
SELECT shop, item, qty FROM sales
```

**リストで渡した順に並ぶ。** 列の並べ替えも同時にできる。

```python
df[["qty", "shop"]]     # この順になる
```

列を捨てる書き方もある。

```python
df.drop(columns=["price"])      # price 以外
```

残す列が少なければ `df[[...]]`、捨てる列が少なければ `drop`。

---
## 4. 行を絞る (WHERE)

In [ ]:
df[df["shop"] == "渋谷"]

In [ ]:
# 中で何が起きているか。条件式は True/False の Series になる
mask = df["shop"] == "渋谷"
print(mask.tolist())
print()
print("それを [] に渡すと、True の行だけ残る")
df[mask]

In [ ]:
# 条件を複数つなぐ。& (かつ) | (または) ~ (でない)
# それぞれの条件を必ず括弧で囲む
df[(df["shop"] == "渋谷") & (df["qty"] >= 2)]

In [ ]:
# よく使う書き方
display(df[df["shop"].isin(["渋谷", "横浜"])])   # IN
display(df[df["qty"].between(2, 3)])             # BETWEEN
df[df["item"].str.contains("コーヒー")]           # LIKE '%コーヒー%'

```sql
SELECT * FROM sales WHERE shop = '渋谷' AND qty >= 2
```

| SQL | pandas |
| --- | --- |
| `AND` | `&` |
| `OR` | `\|` |
| `NOT` | `~` |
| `IN (...)` | `.isin([...])` |
| `BETWEEN a AND b` | `.between(a, b)` |
| `LIKE '%x%'` | `.str.contains("x")` |

> [!IMPORTANT]
> **`and` `or` `not` は使えない。** `&` `|` `~` を使い、
> **各条件を括弧で囲む**。これを忘れると意味不明なエラーが出る。

---
## 5. 列を作る (SELECT ... AS)

In [ ]:
df["total"] = df["qty"] * df["price"]
df

```sql
SELECT *, qty * price AS total FROM sales
```

**列同士の計算は、行ごとに勝手に対応してくれる。** ループは要らない。

```python
df["total"] = df["qty"] * df["price"]      # 全行まとめて
```

これを**ベクトル化**という。pandas でループを書きたくなったら、
だいたい書かずに済む方法がある。

In [ ]:
# 文字列の列も同じ。.str を挟むと文字列操作ができる
df["label"] = df["shop"] + "/" + df["item"]
df[["sale_id", "label"]].head(4)

In [ ]:
# 条件で値を分ける
import numpy as np
df["size"] = np.where(df["qty"] >= 3, "大口", "通常")
df[["sale_id", "qty", "size"]]

---
## 6. 並べる (ORDER BY)

In [ ]:
df.sort_values("total", ascending=False)

In [ ]:
# 複数キー。向きも列ごとに変えられる
df.sort_values(["shop", "total"], ascending=[True, False])

```sql
SELECT * FROM sales ORDER BY shop ASC, total DESC
```

- `ascending=True` が既定(昇順)
- 複数列を指定するときはリスト。`ascending` もリストで渡せる
- **元の `df` は変わらない。** 並べ替えた**新しい表が返る**

`sort_values` の後に index が飛び番になるのが気になるなら
`.reset_index(drop=True)` を足す。

---
## 7. まとめる (GROUP BY)

**pandas でいちばんよく使う操作。**

In [ ]:
df.groupby("shop", as_index=False).agg(
    n_sales=("sale_id", "size"),
    total=("total", "sum"),
)

```sql
SELECT shop, count(*) AS n_sales, sum(total) AS total
FROM sales GROUP BY shop
```

書き方はこの形で覚える。

```python
df.groupby(キー, as_index=False).agg(
    出したい列名=(元の列, 集約関数),
    ...
)
```

| 集約関数 | 意味 |
| --- | --- |
| `"size"` | 行数 (`count(*)`) |
| `"count"` | 欠損でない数 (`count(col)`) |
| `"sum"` | 合計 |
| `"mean"` | 平均 |
| `"min"` / `"max"` | 最小 / 最大 |
| `"nunique"` | 種類数 (`count(distinct)`) |

`as_index=False` を付けると、キーが普通の列として残る。
付けないとキーが index に入って扱いづらい。**とりあえず付けておく。**

In [ ]:
# キーは複数指定できる
df.groupby(["shop", "item"], as_index=False).agg(
    qty=("qty", "sum"),
    total=("total", "sum"),
)

In [ ]:
# 1列だけでよければ短く書ける
display(df.groupby("item")["total"].sum())
df.groupby("item")["total"].sum().reset_index()

---
## 8. 横にくっつける (JOIN)

In [ ]:
display(shops)
df.merge(shops, on="shop", how="left")

```sql
SELECT * FROM sales LEFT JOIN shops ON sales.shop = shops.shop
```

```python
左.merge(右, on=キー, how=結合の種類)
```

| `how` | 意味 |
| --- | --- |
| `"inner"` | 両方にある行だけ(既定) |
| `"left"` | **左を全部残す**。よく使う |
| `"right"` | 右を全部残す |
| `"outer"` | 両方を全部残す |

キーの列名が左右で違うときは `left_on` / `right_on`。

```python
df.merge(shops, left_on="shop", right_on="shop_name", how="left")
```

> [!IMPORTANT]
> **merge の前後で行数を必ず確認する。**
> 右側にキーの重複があると、**行が増える**。
> `inner` にすると、結合できなかった行が**黙って消える**。

In [ ]:
# 行数が変わっていないかを確かめる癖をつける
merged = df.merge(shops, on="shop", how="left")
print("結合前:", len(df))
print("結合後:", len(merged))

# 大宮は売上が無いので、left join では出てこない
print("結合できなかった行:", merged["area"].isna().sum())

---
## 9. 縦にくっつける (UNION ALL)

In [ ]:
jan = df[df["sale_date"] <= "2024-01-07"]
feb = df[df["sale_date"] > "2024-01-07"]
print(len(jan), len(feb))

both = pd.concat([jan, feb], ignore_index=True)
print(len(both))
both.head(3)

```sql
SELECT * FROM a UNION ALL SELECT * FROM b
```

```python
pd.concat([df1, df2, df3], ignore_index=True)
```

- **リストで渡す。** `df1.concat(df2)` とは書かない
- **列名で揃う。** 列の順番が違っても正しくくっつく。
  片方にしか無い列は欠損で埋まる
- `ignore_index=True` で index を振り直す。付けないと `0,1,2,0,1,2` のように重複する

複数ファイルを読んで1つにするときの定番。

```python
frames = [pd.read_csv(p) for p in paths]
df = pd.concat(frames, ignore_index=True)
```

---
## 10. 重複を消す (DISTINCT)

In [ ]:
# 全列が一致する行を1つにする
print(len(both), "→", len(both.drop_duplicates()))

# キーを指定すると、その列が同じ行を1つにする
display(df[["shop", "item"]].drop_duplicates())

In [ ]:
# どれを残すかを選べる
d = pd.DataFrame({"k": ["a", "a", "b"], "v": [1, 2, 3]})
display(d)
display(d.drop_duplicates("k", keep="first"))   # 最初を残す (既定)
d.drop_duplicates("k", keep="last")             # 最後を残す

```sql
SELECT DISTINCT shop, item FROM sales
```

```python
df.drop_duplicates()                          # 全列一致
df.drop_duplicates(subset="key")              # キーで
df.drop_duplicates(subset="key", keep="last") # 最後を残す
```

**「キーごとに最新の1行を残す」**はこう書く。並べてから最後を残す。

```python
df.sort_values("updated_at").drop_duplicates("id", keep="last")
```

実務で非常によく出る形。`pandas-03` で詳しくやる。

---
## 11. 書く

In [ ]:
df.to_csv("/tmp/out.csv", index=False)
df.to_parquet("/tmp/out.parquet", index=False)

print(open("/tmp/out.csv").read()[:200])

```python
df.to_csv(path, index=False)
df.to_parquet(path, index=False)
```

**`index=False` を付ける。** 付けないと、意味のない連番が
1列目に書き出されて、次に読んだときに `Unnamed: 0` という列になる。

保存は基本 Parquet。CSV より小さく、速く、**型が保存される**。
人が目で見るファイルだけ CSV にする。

---
## つなげる

ここまでの操作は**返り値が新しい DataFrame** なので、`.` でつないで書ける。

In [ ]:
result = (
    df
    .query("qty >= 2")                                    # WHERE
    .groupby("shop", as_index=False)                      # GROUP BY
    .agg(n=("sale_id", "size"), total=("total", "sum"))   # 集約
    .sort_values("total", ascending=False)                # ORDER BY
    .reset_index(drop=True)
)
result

```sql
SELECT shop, count(*) AS n, sum(total) AS total
FROM sales WHERE qty >= 2
GROUP BY shop ORDER BY total DESC
```

SQL とほぼ同じ順に、上から読める。

**全体を括弧で囲む**と、行末のバックスラッシュが要らなくなる。
書き方の作法として覚えておくと楽。

長いチェーンで詰まったら、途中で切って `display()` する。
どこで想定と変わったかがすぐ分かる。

---
## 練習

10個を使うだけ。ひねりは無い。

In [ ]:
# 練習1: item ごとの売上合計を出す。列名は item, total。合計の大きい順。

sales = pd.read_csv("/data/sales.csv")
sales["total"] = sales["qty"] * sales["price"]

ans = ...   # ここに書く

assert list(ans.columns) == ["item", "total"], list(ans.columns)
assert ans["item"].tolist() == ["コーヒー", "ケーキ", "紅茶"], ans["item"].tolist()
assert ans["total"].tolist() == [5400, 4200, 1600]
print("OK")

In [ ]:
# 練習2: 東京にある店の売上だけを合計する。(shops と結合してから絞る)

shops = pd.read_csv("/data/shops.csv")

ans = ...   # ここに書く (数値)

assert ans == 9150, f"9150 のはず: {ans}"
print("OK")

In [ ]:
# 練習3: 店ごとに、売上合計と、扱った商品の種類数を出す。
#        列名は shop, total, n_items。shop の昇順。

ans = ...   # ここに書く

assert list(ans.columns) == ["shop", "total", "n_items"], list(ans.columns)
assert ans["shop"].tolist() == ["新宿", "横浜", "渋谷"], ans["shop"].tolist()
assert ans["total"].tolist() == [3650, 2050, 5500]
assert ans["n_items"].tolist() == [3, 3, 3]
print("OK")

---
## 早見表

| # | やること | SQL | pandas |
| --- | --- | --- | --- |
| 1 | 読む | — | `pd.read_csv(path)` |
| 2 | 見る | — | `df.head()` / `df.shape` / `df.dtypes` / `df["c"].unique()` / `df["c"].value_counts()` |
| 3 | 列を選ぶ | `SELECT a, b` | `df[["a", "b"]]` |
| 4 | 行を絞る | `WHERE` | `df[(df["a"] == 1) & (df["b"] > 2)]` |
| 5 | 列を作る | `... AS x` | `df["x"] = df["a"] * df["b"]` |
| 6 | 並べる | `ORDER BY` | `df.sort_values("a", ascending=False)` |
| 7 | まとめる | `GROUP BY` | `df.groupby("k", as_index=False).agg(x=("a", "sum"))` |
| 8 | 横に結合 | `JOIN` | `df.merge(other, on="k", how="left")` |
| 9 | 縦に結合 | `UNION ALL` | `pd.concat([a, b], ignore_index=True)` |
| 10 | 重複を消す | `DISTINCT` | `df.drop_duplicates(subset="k", keep="last")` |
| — | 書く | — | `df.to_parquet(path, index=False)` |

### 忘れがちな3つ

- **条件は `&` `|` `~`。各条件を括弧で囲む。** `and` は使えない
- **`groupby` には `as_index=False`。** キーを列のまま残す
- **`to_csv` / `to_parquet` には `index=False`**

### 数を数える癖

| 操作 | 確かめること |
| --- | --- |
| `merge` の後 | 行数が増えていないか、減っていないか |
| `groupby` の後 | 合計が元と一致するか |
| `drop_duplicates` の後 | 減った数に説明が付くか |
| 条件で絞った後 | 落ちた行が意図どおりか |

**行数が変わる操作の前後では必ず `len(df)` を見る。** これだけで事故の大半は防げる。

---

次: `pandas-01-read-and-types.ipynb`(引数の違いで何が変わるか)